# Kaggle runner — E1 separated feature selection + classification (precomputed 5×5) — survey v3 (QĐ15)

**v3:** 5 bộ LUAD vs LUSC (TCGA trong `experiment-data-perm`, 4 cohort GEO trong `experiment-data-geo-target`), k ∈ {1,…,50}; pha B chỉ LR-L2 (+ RF, SVM ở k ∈ {5, 25}), B0 và random-k; wrapper chỉ 3 cohort GPL570, 3×5, k ∈ {5, 10, 50}. Bản chia `repeated_splits.csv` phải sinh lại bằng `build_e1_repeated_splits.py` cho 5 bộ.

Notebook dùng **hai input Kaggle Dataset**: `experiment-data-perm` chứa expression/metadata và `e1_repeated_splits_5x5` chứa ba `repeated_splits.csv`. CSV chỉ định 25 outer train/test split (5 repeat × 5 fold). Feature selection chỉ thấy train; phân loại tune inner-CV trong train và chấm outer-test đúng một lần.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

INPUT_DATASET = Path('/kaggle/input/experiment-data-perm')       # survey v3: cần cả experiment-data-perm/ (TCGA) và experiment-data-geo-target/ (4 cohort GEO) -> xem INPUT_GEO_DATASET
INPUT_SPLITS_DATASET = Path('/kaggle/input/e1-repeated-splits-5x5')  # sửa nếu Kaggle mount slug khác
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
INPUT_GEO_DATASET = Path('/kaggle/input/experiment-data-geo-target')  # sửa theo slug Dataset Kaggle
COHORTS = ['tcga_luad_lusc_primary_tumor', 'gse10245', 'gse30219', 'gse37745', 'gse42127']   # survey v3 (QĐ15)
GEO_DIRS = {'gse10245': 'gse10245', 'gse30219': 'gse30219_r5_fixed', 'gse37745': 'gse37745', 'gse42127': 'gse42127'}
K_GRID = [1, 2, 3, 4, 5, 10, 25, 50]   # survey v3
RUN_FILTER_EMBEDDED = True   # False nếu chỉ muốn chạy GA/BPSO
RUN_FULL_CLASSIFIERS = True  # pha B: LR-L2 (+ RF, SVM ở k ∈ {5, 25}); bộ 11 mô hình chỉ với --models-extended (phụ lục)
RUN_WRAPPERS = False         # GA/BPSO là job tách riêng
RUN_WRAPPER_CLASSIFIERS = True  # classifier cho panel GA/BPSO
# RUN_WRAPPERS: một Save Version = một cohort, chạy lần lượt GA và BPSO.
TARGET_COHORT = 'gse37745'   # wrapper CHỈ 3 cohort GPL570: gse10245, gse30219, gse37745 (WRAPPER_DATASETS)
TARGET_WRAPPER_METHODS = ['ga', 'bpso']
TARGET_REPEATS = [0, 1, 2]  # wrapper 3×5 (WRAPPER_REPEATS); k = WRAPPER_K = [5, 10, 50]
WRAPPER_K = [5, 10, 50]
# Nếu tiếp tục một job dang dở: attach ZIP output của Version trước và điền đường dẫn này.
PREVIOUS_PROGRESS_ARCHIVE = None  # ví dụ Path('/kaggle/input/e1-gse-output/e1_gse68465_stage_early_late_wrappers_output.zip')


In [ ]:
# Gate 1: xác nhận hai input và 25 split cố định/cohort trước khi chạy.
assert INPUT_DATASET.is_dir(), f'Không thấy expression input: {INPUT_DATASET}'
assert INPUT_SPLITS_DATASET.is_dir(), f'Không thấy split input: {INPUT_SPLITS_DATASET}'
assert INPUT_GEO_DATASET.is_dir(), f'Không thấy GEO input: {INPUT_GEO_DATASET}'
for cohort in COHORTS:
    root = (INPUT_GEO_DATASET / GEO_DIRS[cohort]) if cohort in GEO_DIRS else (INPUT_DATASET / cohort)
    for filename in ('expression.h5', 'metadata.csv'):
        assert (root / filename).is_file(), f'Thiếu {cohort}/{filename}'
    split_path = INPUT_SPLITS_DATASET / cohort / 'repeated_splits.csv'
    assert split_path.is_file(), f'Thiếu split: {split_path}'
    split = pd.read_csv(split_path)
    assert set(split.role) == {'train', 'test'}
    assert split[['repeat', 'fold']].drop_duplicates().shape[0] == 25
    assert (split[split.role == 'test'].groupby(['repeat', 'sample_id']).size() == 1).all()
    assert not (split[split.role == 'train'].merge(split[split.role == 'test'], on=['repeat', 'fold', 'patient_id'])).shape[0]
print('INPUT_AND_SPLITS_OK')


In [ ]:
# Gate 2: clone đúng commit đã push, cài dependencies, rồi liên kết input read-only (không copy H5).
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if PREVIOUS_PROGRESS_ARCHIVE is not None:
    assert PREVIOUS_PROGRESS_ARCHIVE.is_file(), f'Không thấy archive resume: {PREVIOUS_PROGRESS_ARCHIVE}'
    shutil.unpack_archive(str(PREVIOUS_PROGRESS_ARCHIVE), str(WORK_ROOT))
    print('RESTORED_PROGRESS', PREVIOUS_PROGRESS_ARCHIVE)
link = WORK_ROOT / 'experiment-data-perm'
if not link.exists(): link.symlink_to(INPUT_DATASET, target_is_directory=True)
assert link.resolve() == INPUT_DATASET.resolve()
link_geo = WORK_ROOT / 'experiment-data-geo-target'            # survey v3: 4 cohort GEO
if not link_geo.exists(): link_geo.symlink_to(INPUT_GEO_DATASET, target_is_directory=True)
assert link_geo.resolve() == INPUT_GEO_DATASET.resolve()
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
print('COMMIT', commit_hash)


In [ ]:
# Phase A — filter + embedded. Không có classifier ở cell này.
if RUN_FILTER_EMBEDDED:
    subprocess.run([sys.executable, 'scripts/run_e1_feature_selection.py', '--datasets', *COHORTS, '--k', *map(str, K_GRID), '--splits-root', str(INPUT_SPLITS_DATASET)], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Phase B — đọc fixed panels của Phase A; inner-CV chỉ dùng outer train.
if RUN_FILTER_EMBEDDED and RUN_FULL_CLASSIFIERS:
    subprocess.run([sys.executable, 'scripts/run_e1_classification.py', '--datasets', *COHORTS, '--secondary', '--splits-root', str(INPUT_SPLITS_DATASET)], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Job RIÊNG: mỗi Save Version chạy GA+BPSO cho một cohort; checkpoint sau mỗi fold.
if RUN_WRAPPERS:
    subprocess.run([sys.executable, 'scripts/run_e1_wrappers_all_cohorts.py', '--datasets', TARGET_COHORT, '--methods', *TARGET_WRAPPER_METHODS, '--repeats', *map(str, TARGET_REPEATS), '--k', *map(str, WRAPPER_K), '--splits-root', str(INPUT_SPLITS_DATASET)], cwd=REPO_DIR, env=os.environ.copy(), check=True)
    if RUN_WRAPPER_CLASSIFIERS:
        subprocess.run([sys.executable, 'scripts/run_e1_classification.py', '--datasets', TARGET_COHORT, '--methods', *TARGET_WRAPPER_METHODS, '--secondary', '--no-references', '--splits-root', str(INPUT_SPLITS_DATASET), '--selection-roots', str(WORK_ROOT / 'results/survey_v2/E1_wrapper_feature_selection')], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Gate 3: chỉ save/download output nếu provenance đầy đủ.
result_root = WORK_ROOT / 'results/survey_v2'
configs = list(result_root.glob('E1_*/*/config.json'))
assert configs, 'Không tìm thấy config output'
for path in configs:
    cfg = json.loads(path.read_text(encoding='utf-8'))
    assert cfg.get('traceable') is True and cfg.get('commit_hash') == commit_hash, path
archive = Path('/kaggle/working') / f'e1_{TARGET_COHORT}_wrappers_output.zip'
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results')
assert archive.is_file() and archive.stat().st_size > 0
print('E1_OK', result_root, 'ARCHIVE=', archive)
